# OurAirports Data Cleansing & Merge

**Purpose**: merge `airports-ourairports.csv` with `runways.csv` (the list attribute for requirement 6), do light cleaning, and save one integrated OurAirports-side dataset to `data/interim/`.

Unlike Wikidata, there's no query-reliability problem here (it's a flat CSV download), so there's no separate profiling notebook needed — this single notebook covers merge + cleaning.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

RAW_DIR = project_root / "data" / "raw" / "ourairports"
INTERIM_DIR = project_root / "data" / "interim"

airports = pd.read_csv(RAW_DIR / "airports-ourairports.csv")
runways = pd.read_csv(RAW_DIR / "runways.csv")

print(f"airports: {airports.shape}")
print(f"runways:  {runways.shape}")

airports: (86151, 19)
runways:  (48278, 20)


## `type` overview

Not filtering out `closed` airports here -- that's a decision for the matching/fusion stage, not cleansing. Keeping the full set and the `type` column so it's available as a filter later.

In [2]:
airports["type"].value_counts()

type
small_airport     42762
heliport          23224
closed            13548
medium_airport     4106
seaplane_base      1273
large_airport      1175
balloonport          63
Name: count, dtype: int64

## Aggregate runways into a list attribute

Confirmed in `requirement_analysis.ipynb`: `runways.airport_ident` joins cleanly to `airports.ident` (no orphans), 41,163 airports have at least one runway, ~13% have more than one. Each airport gets a `runways` column holding a list of dicts (one per runway) instead of flattening into separate rows -- that's what makes this the list attribute for requirement 6.

In [8]:
RUNWAY_FIELDS = ["airport_ident", "length_ft", "width_ft", "surface", "lighted", "closed", "le_ident", "he_ident"]

runways_sub = (
    runways[RUNWAY_FIELDS]
    .rename(
        columns = {
            "airport_ident": "ident"
        }
    )
    )
runways_sub.head()

,ident,length_ft,width_ft,surface,lighted,closed,le_ident,he_ident
0,00A,80.0,80.0,ASPH-G,1,0,H1,NaN
1,00AK,2500.0,40.0,GRVL,0,0,N,S
2,00AL,2100.0,90.0,TURF,0,0,01,19
3,00AN,4517.0,60.0,GVL,0,0,3,21
4,00AS,1450.0,60.0,Turf,0,0,1,19


In [11]:
result = pd.merge(
    left = airports,
    right = runways_sub,
    on = "ident",
    how = "left"
)
result.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,...,home_link,wikipedia_link,keywords,length_ft,width_ft,surface,lighted,closed,le_ident,he_ident
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,...,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN,80.0,80.0,ASPH-G,1.0,0.0,H1,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,...,NaN,NaN,NaN,2500.0,40.0,GRVL,0.0,0.0,N,S
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,...,NaN,NaN,NaN,2100.0,90.0,TURF,0.0,0.0,01,19
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,...,NaN,NaN,NaN,4517.0,60.0,GVL,0.0,0.0,3,21


## Merge + trim irrelevant columns

Dropping only `keywords` -- free-text tags, high-noise, no clear integration value. Keeping `home_link` and, importantly, `wikipedia_link`: OurAirports links each airport directly to its Wikipedia article, and Wikidata items generally carry a Wikipedia sitelink (DBpedia resource URIs are literally derived from the article title) -- so this is a potential high-precision, near-exact matching key against Wikidata for identity resolution later. Dropping it earlier was a mistake.

In [18]:
DROP_COLUMNS = ["keywords"]

final_df = result.drop(columns=DROP_COLUMNS)

print(final_df.shape)

print(final_df.duplicated(subset = ['ident']).sum())
final_df.head(3)

(93266, 25)
7115


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,...,local_code,home_link,wikipedia_link,length_ft,width_ft,surface,lighted,closed,le_ident,he_ident
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,...,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,80.0,80.0,ASPH-G,1.0,0.0,H1,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,...,00AA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,...,00AK,NaN,NaN,2500.0,40.0,GRVL,0.0,0.0,N,S


In [19]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
out_path = INTERIM_DIR / "ourairports_airports.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved {len(final_df):,} airport - runway tuples to {out_path}")

Saved 93,266 airport - runway tuples to /Users/huanglinchun/dev/studies/uni-mannheim/Web-Data-Integration/Web-Data-Integration-Project/data/interim/ourairports_airports.csv


## Notes

- `runways` is saved as a Python-list-of-dicts literal inside a CSV cell (via pandas' default `str()` repr) -- fine for now, but re-reading this CSV later needs `ast.literal_eval` on that column, not `pd.read_csv` alone. If this friction becomes annoying, switch to Parquet, which preserves nested types natively.
- `closed` airports (13,548 of them) are still in the dataset -- filtering, if wanted, belongs at the matching/fusion stage, not here.
- No profiling notebook for this source: it's a flat CSV, not a queried API, so there's no query-reliability story to document the way there was for Wikidata/QLever.
- `wikipedia_link` is kept (19.5% coverage, 16,757 airports) as a planned high-precision matching key against Wikidata during identity resolution.
- OurAirports' data provenance is largely independent of Wikidata/DBpedia (mostly FAA/DAFIF + community + specialist aviation databases, per ourairports.com/about.html), so this isn't circular, it's just two independently-maintained sources that both happen to reference the same Wikipedia article for a given airport.